# PolicyPilot: long context vs RAG on a free Kaggle GPU

Runs the pre-registered grid in `docs/LONG_CONTEXT_PLAN.md`: Qwen2.5-7B-Instruct (4-bit AWQ, 32K window)
served by vLLM with prefix caching, reading the policy library in context, against single-shot RAG
with the same model. No paid API and no keys.

**Settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **on**.

**Time:** about 1,000 model calls, roughly 6 to 10 hours on 2 x T4 (an estimate, not measured).
Every requirement is checkpointed to `/kaggle/working/long_context_runs.jsonl`, so the best way to
run it is **Save Version → Save & Run All**: it runs in the background for up to 12 hours. If it
stops before the end, add that version's output as an input to the next version (step 1 picks up
the checkpoint) and run again.

When it is done, download `long_context_runs.jsonl` and put it in the repo at
`data/eval/long_context_runs.jsonl`, then run `python -m policypilot.longctx report`.

In [ ]:
# 1. Get the code, and any checkpoint from an earlier version of this notebook
import glob, os, shutil, subprocess
REPO, BRANCH = "https://github.com/vik-bit/policypilot.git", "long-context"
if not os.path.exists("/kaggle/working/policypilot"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/kaggle/working/policypilot"], check=True)
os.chdir("/kaggle/working/policypilot")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

RUNS = "/kaggle/working/long_context_runs.jsonl"
earlier = [p for p in glob.glob("/kaggle/input/**/long_context_runs.jsonl", recursive=True)]
if earlier and not os.path.exists(RUNS):
    shutil.copy(earlier[0], RUNS)
    print("Resuming from", earlier[0], sum(1 for _ in open(RUNS)), "rows")

In [ ]:
# 2. Install. vLLM brings its own torch; PolicyPilot is installed without its web/agent extras,
#    which long-context mode does not import.
!pip install -q "vllm==0.6.6.post1"
!pip install -q --no-deps -e .
!pip install -q "fastembed>=0.4" "tokenizers>=0.20" pypdf

In [ ]:
# 3. Start vLLM in the background and wait until it answers.
import subprocess, time, urllib.request
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

def start(extra):
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--quantization", "awq",
           "--dtype", "half", "--max-model-len", "32768", "--tensor-parallel-size", "2",
           "--gpu-memory-utilization", "0.90", "--enable-prefix-caching", "--port", "8000", *extra]
    return subprocess.Popen(cmd, stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT)

def wait(proc, minutes=20):
    for _ in range(minutes * 12):
        if proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            return True
        except OSError:
            time.sleep(5)
    return False

# Ask vLLM to report cached prompt tokens; older builds without the flag are started without it.
server = start(["--enable-prompt-tokens-details"])
if not wait(server):
    server.kill(); server = start([])
    assert wait(server), open("/kaggle/working/vllm.log").read()[-3000:]
print("vLLM is up")

In [ ]:
# 4. Exact prompt sizes with the Qwen tokenizer (no model call)
!python -m policypilot.longctx tokens --distractors 0 50 133

In [ ]:
# 5. Smoke check on 5 requirements of the development set (not the held-out set)
!python -m policypilot.longctx --model {MODEL} eval --grid smoke --out /kaggle/working/smoke_runs.jsonl
!tail -n 2 /kaggle/working/smoke_runs.jsonl | cut -c 1-600

In [ ]:
# 6. The pre-registered grid on the held-out `real` set. Safe to re-run: finished rows are skipped.
!python -m policypilot.longctx --model {MODEL} eval --grid main --out {RUNS}

In [ ]:
# 7. Tables, intervals and paired tests (no model call)
shutil.copy(RUNS, "data/eval/long_context_runs.jsonl")
!python -m policypilot.longctx report
shutil.copy("docs/LONG_CONTEXT_RESULTS.md", "/kaggle/working/LONG_CONTEXT_RESULTS.md")
shutil.copy("data/eval/long_context_results.json", "/kaggle/working/long_context_results.json")